In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
!pip -q uninstall -y bitsandbytes
!pip -q install -U wandb
!pip install sacrebleu

# 1.学習コード

## 1-1.Config

In [17]:
import os, re, math, random
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Any

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    set_seed,
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# HF上のQwen 4Bを指定（あなたの推論コードの雰囲気に合わせるならQwen3）
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"   # 例（必要なら差し替え）
# MODEL_NAME = "Qwen/Qwen3-4B"               # ベースを使う場合はこちら等

TRAIN_CSV = "/content/drive/MyDrive/kaggle/Akkadian/data/train.csv"
OUTPUT_DIR = "/content/drive/MyDrive/kaggle/Akkadian/models/qwen3-4b-lora-deeppast-epoch3"

SEED = 42
VAL_RATIO = 0.02

MAX_LEN = 1024
MICRO_BS = 1
GRAD_ACCUM = 16


WARMUP_RATIO = 0.03
SAVE_STEPS = 10
EVAL_STEPS = 10
LOG_STEPS = 10

# QLoRA（4bit）で省メモリ学習（Kaggle T4等向け）
USE_4BIT = False

# LoRA設定（Qwen系の定番 target_modules）
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LR = 5e-5   # 2e-4 → 5e-5 に
EPOCHS = 3  # 1→2（ただしval loss見て）

TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
set_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True

## 1-2 wandbの設定

In [18]:
import wandb
wandb.login()


RUN_NAME = "qwen3-4b-lora-deeppast-epoch3"
wandb.init(
    project="deep-past-akkadian",
    name=RUN_NAME,
    config={
        "model": MODEL_NAME,
        "seed": SEED,
        "val_ratio": VAL_RATIO,
        "max_len": MAX_LEN,
        "micro_bs": MICRO_BS,
        "grad_accum": GRAD_ACCUM,
        "epochs": EPOCHS,
        "lr": LR,
        "warmup_ratio": WARMUP_RATIO,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "use_4bit": USE_4BIT,
    },
)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


val/bleu,▁
val/chrf++,▁
val/geo_mean,▁
val/bleu,11.76976
val/chrf++,38.56963
val/geo_mean,21.30622


## 1-3 Helper

In [19]:
# ---------- [Cell 3] Normalization (あなたの推論ノートと同等) ----------
def normalize_gaps(text):
    """
    Normalize ALL gap variants to a single <gap> token.
    Adjacent gaps are NOT merged.
    """
    if not isinstance(text, str):
        return str(text) if text else ""

    text = re.sub(r'\((?:large\s+)?break\)', '<gap>', text, flags=re.I)
    text = re.sub(r'\(\d+\s+broken\s+lines?\)', '<gap>', text, flags=re.I)

    text = re.sub(r'\.{3,}', ' <gap> ', text)
    text = re.sub(r'……', ' <gap> ', text)
    text = re.sub(r'…', ' <gap> ', text)

    text = re.sub(r'\[x\]', '<gap>', text, flags=re.I)
    text = re.sub(r'\bxx+\b', ' <gap> ', text)
    text = re.sub(r'\bx\b', '<gap>', text)
    return text


def normalize_transliteration(text):
    if pd.isna(text):
        return ""
    text = str(text)

    text = normalize_gaps(text)
    text = text.replace('ḫ', 'h').replace('Ḫ', 'H')

    subscript_to_normal = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    text = text.translate(subscript_to_normal)

    text = text.replace('KÙ.B.', 'KÙ.BABBAR')
    text = re.sub(r'(\d+\.\d{4})\d+', r'\1', text)

    for pat, frac in [
        (r'\b0\.8333\d*\b', '⅚'), (r'\b0\.1666\d*\b', '⅙'),
        (r'\b0\.6666\d*\b', '⅔'), (r'\b0\.3333\d*\b', '⅓'),
        (r'\b0\.625\b', '⅝'),
        (r'\b0\.75\b', '¾'), (r'\b0\.25\b', '¼'), (r'\b0\.5\b', '½'),
    ]:
        text = re.sub(pat, frac, text)

    text = re.sub(r'\s+', ' ', text).strip()
    return text


def postprocess_output(text):
    """
    参考：あなたの推論コードの postprocess_output（学習ターゲット側にも適用可）
    """
    if not isinstance(text, str):
        return str(text) if text else "broken text"
    if not text.strip():
        return "broken text"

    text = re.sub(r'\[x\]', '<gap>', text, flags=re.IGNORECASE)
    text = re.sub(r'\(x\)', '<gap>', text, flags=re.IGNORECASE)
    text = re.sub(r'(\.{3,}|…|\[\.+\])', '<gap>', text)
    text = re.sub(r'\((?:large\s+)?break\)', '<gap>', text, flags=re.I)
    text = re.sub(r'\(\d+\s+broken\s+lines?\)', '<gap>', text, flags=re.I)
    text = re.sub(r'\bx\b', '<gap>', text)

    text = re.sub(r'\(fem\.?\s*(?:plur\.?|pl\.?|sing\.?|singular|plural)?\)', '', text, flags=re.I)
    text = re.sub(r'\((?:plur|pl|sing|singular|plural)\.?\)', '', text, flags=re.I)
    text = re.sub(r'\(\?\)', '', text)
    text = re.sub(r'\(!\)', '', text)
    text = re.sub(r'\bfem\.\s*', '', text, flags=re.I)
    text = re.sub(r'\bsing\.\s*', '', text, flags=re.I)
    text = re.sub(r'\bpl\.\s*', '', text, flags=re.I)
    text = re.sub(r'\bplural\b\s*', '', text, flags=re.I)

    text = re.sub(r'\(d\)', '{d}', text)
    text = re.sub(r'\(ki\)', '{ki}', text)
    text = re.sub(r'\(TÚG\)', 'TÚG', text)

    text = re.sub(r'\bPN\b', '<gap>', text)
    text = text.replace('-gold', 'pašallum gold')
    text = text.replace('-tax', 'šadduātum tax')
    text = re.sub(r'(?<!kutānum )\btextiles\b', 'kutānum textiles', text)

    text = re.sub(r'\b1\s*/\s*12\s*\(?\s*shekel\s*\)?', '⅔ shekel 15 grains', text, flags=re.I)
    text = re.sub(r'\b5\s+11\s*/\s*12\s+shekels?', '6 shekels less 15 grains', text, flags=re.I)
    text = re.sub(r'\b7\s*/\s*12\s+shekels?', '½ shekel 15 grains', text, flags=re.I)
    text = re.sub(r'\b5\s*/\s*12\s+shekels?', '15 grains', text, flags=re.I)

    text = re.sub(r'(\w+)\s+/\s+(\w+)', r'\1', text)

    for roman, integer in [
        ('XII', '12'), ('VIII', '8'), ('XI', '11'), ('VII', '7'),
        ('VI', '6'), ('IX', '9'), ('IV', '4'), ('III', '3'),
        ('II', '2'), ('X', '10'), ('V', '5'), ('I', '1'),
    ]:
        text = re.sub(r'\b(month)\s+' + roman + r'\b', r'\1 ' + integer, text, flags=re.I)

    text = re.sub(r'(\d+\.\d{4})\d+', r'\1', text)
    text = text.replace('ḫ', 'h').replace('Ḫ', 'H')

    subscript_map = {'₀':'0','₁':'1','₂':'2','₃':'3','₄':'4','₅':'5','₆':'6','₇':'7','₈':'8','₉':'9'}
    for sub, normal in subscript_map.items():
        text = text.replace(sub, normal)

    text = text.replace('<gap>', '\x00GAP\x00')
    forbidden_chars = '()—–<>⌈⌊⌋[]+ʾ/;'
    for char in forbidden_chars:
        text = text.replace(char, '')
    text = re.sub(r'<<\s*>>', '', text)
    text = text.replace('\x00GAP\x00', '<gap>')

    for pattern, replacement in [
        (r'(\d+)\.8333\d*\b', r'\1 ⅚'), (r'\b0\.8333\d*\b', '⅚'),
        (r'(\d+)\.1666\d*\b', r'\1 ⅙'), (r'\b0\.1666\d*\b', '⅙'),
        (r'(\d+)\.6666\d*\b', r'\1 ⅔'), (r'\b0\.6666\d*\b', '⅔'),
        (r'(\d+)\.3333\d*\b', r'\1 ⅓'), (r'\b0\.3333\d*\b', '⅓'),
        (r'(\d+)\.625\b', r'\1 ⅝'), (r'\b0\.625\b', '⅝'),
        (r'(\d+)\.75\b', r'\1 ¾'), (r'\b0\.75\b', '¾'),
        (r'(\d+)\.25\b', r'\1 ¼'), (r'\b0\.25\b', '¼'),
        (r'(\d+)\.5\b', r'\1 ½'), (r'\b0\.5\b', '½'),
    ]:
        text = re.sub(pattern, replacement, text)

    text = text.replace(' 1 2', ' ½').replace(' 1 3', ' ⅓').replace(' 2 3', ' ⅔')
    text = text.replace(' 1 4', ' ¼').replace(' 3 4', ' ¾')

    text = re.sub(r'\.\..+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'^-\s*', '', text)
    text = re.sub(r'\s*-$', '', text)

    return text if text else "broken text"

## 1-4 学習

In [20]:
# ---------- [Cell 4] Load tokenizer/model from Hugging Face ----------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
    trust_remote_code=True,
)

# 省メモリ＆学習安定化
model.config.use_cache = False
model.gradient_checkpointing_enable()

# QLoRA向けの下準備
if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


# ---------- [Cell 5] Load train.csv -> train/val split ----------
df = pd.read_csv(TRAIN_CSV)
df = df.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

n_val = max(1, int(len(df) * VAL_RATIO))
df_val = df.iloc[:n_val].copy()
df_trn = df.iloc[n_val:].copy()

train_ds = Dataset.from_pandas(df_trn, preserve_index=False)
val_ds   = Dataset.from_pandas(df_val, preserve_index=False)


# ---------- [Cell 6] Tokenize with chat_template + "completion-only loss" ----------
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer = getattr(processor, "tokenizer", processor)

def build_texts(translit: str, translation: str):
    user = f"/no_think Translate Akkadian to English: {normalize_transliteration(translit)}"
    assistant = str(translation).strip().replace("\n", " ")

    messages = [{"role":"user","content":user},{"role":"assistant","content":assistant}]
    full_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

    prompt_only = processor.apply_chat_template(
        [{"role":"user","content":user}],
        tokenize=False,
        add_generation_prompt=True
    )
    return prompt_only, full_text


def tokenize_example(ex):
    prompt_text, full_text = build_texts(ex["transliteration"], ex["translation"])

    full = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
    )
    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    input_ids = full["input_ids"]
    attn = full["attention_mask"]

    # promptがprefixになっている前提で境界を取る（崩れてたら最善策で探す）
    boundary = len(prompt_ids)
    if len(input_ids) < boundary or input_ids[:boundary] != prompt_ids:
        boundary = None
        # 先頭付近で探す（通常は0で一致するはず）
        max_search = min(256, len(input_ids))
        for b in range(0, max_search):
            if input_ids[b:b+len(prompt_ids)] == prompt_ids:
                boundary = b + len(prompt_ids)
                break
        if boundary is None:
            boundary = min(len(prompt_ids), len(input_ids))

    labels = [-100] * boundary + input_ids[boundary:]
    labels = labels[:len(input_ids)]  # 念のため

    return {
        "input_ids": input_ids,
        "attention_mask": attn,
        "labels": labels,
    }


train_tok = train_ds.map(tokenize_example, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tokenize_example, remove_columns=val_ds.column_names)


@dataclass
class DataCollatorForCausalLM:
    tokenizer: Any
    pad_to_multiple_of: int = 8

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch = self.tokenizer.pad(
            [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features],
            padding=True,
            return_tensors="pt",
            pad_to_multiple_of=self.pad_to_multiple_of,
        )
        max_len = batch["input_ids"].shape[1]
        labels = []
        for f in features:
            l = f["labels"]
            l = l + [-100] * (max_len - len(l))
            labels.append(l)
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

collator = DataCollatorForCausalLM(tokenizer)


# ---------- [Cell 7] Train ----------
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=MICRO_BS,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",

    logging_steps=LOG_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    report_to="wandb",

    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)

trainer.train()

# LoRAアダプタ保存（推論ノートの adapter_path 側に相当）
trainer.model.save_pretrained(os.path.join(OUTPUT_DIR, "adapter"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "adapter"))


# ---------- [Cell 8] (Optional) Merge LoRA into base and save ----------
# 推論側で merge_and_unload して /tmp/model に保存していたのと同じことをここで実行できます。
from peft import PeftModel

base_for_merge = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

merged = PeftModel.from_pretrained(base_for_merge, os.path.join(OUTPUT_DIR, "adapter"))
merged = merged.merge_and_unload()

MERGED_DIR = os.path.join(OUTPUT_DIR, "merged")
os.makedirs(MERGED_DIR, exist_ok=True)
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print("Saved:")
print(" - adapter:", os.path.join(OUTPUT_DIR, "adapter"))
print(" - merged :", MERGED_DIR)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,1.777115,1.540086
2,1.341378,1.258856
3,1.303324,1.214185


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved:
 - adapter: /content/drive/MyDrive/kaggle/Akkadian/models/qwen3-4b-lora-deeppast-epoch3/adapter
 - merged : /content/drive/MyDrive/kaggle/Akkadian/models/qwen3-4b-lora-deeppast-epoch3/merged


## 1-5 Validation Score (BLEU / chrF++ Geometric Mean)

In [21]:
# ---------- [Cell 9] Evaluate on validation set (BLEU + chrF++ geometric mean) ----------
import math
from transformers import AutoProcessor
from sacrebleu.metrics import BLEU, CHRF
from tqdm import tqdm

# ---- Load merged model from disk (same as section-2 inference) ----
MERGED_DIR = os.path.join(OUTPUT_DIR, "merged")

print("Loading and preparing validation data …")
df = pd.read_csv(TRAIN_CSV)
df = df.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

n_val = max(1, int(len(df) * VAL_RATIO))
df_val = df.iloc[:n_val].copy()
df_trn = df.iloc[n_val:].copy()

train_ds = Dataset.from_pandas(df_trn, preserve_index=False)
val_ds = Dataset.from_pandas(df_val, preserve_index=False)


print("--- Loading merged model for evaluation ---")
eval_processor = AutoProcessor.from_pretrained(MERGED_DIR, trust_remote_code=True)
try:
    eval_tokenizer = eval_processor.tokenizer
except AttributeError:
    eval_tokenizer = eval_processor

eval_tokenizer.padding_side = "left"
if eval_tokenizer.pad_token is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

eval_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
eval_model.eval()

# Stop token IDs (same as section-2 inference)
_eos_id    = eval_tokenizer.eos_token_id
_im_end_id = eval_tokenizer.convert_tokens_to_ids("<|im_end|>")
_stop_ids  = [_eos_id] + ([_im_end_id] if _im_end_id is not None else [])

EVAL_BATCH_SIZE = 8


def _translate_batch(translit_list):
    """Return translated strings for a batch of transliterations."""
    messages_batch = [
        [{"role": "user", "content": f"/no_think Translate Akkadian to English: {normalize_transliteration(t)}"}]
        for t in translit_list
    ]
    raw_prompts = [
        eval_processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in messages_batch
    ]
    inputs = eval_tokenizer(
        raw_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
    ).to(eval_model.device)

    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            top_k=20,
            repetition_penalty=1.05,
            pad_token_id=eval_tokenizer.pad_token_id,
            eos_token_id=_stop_ids,
        )

    input_len = inputs.input_ids.shape[1]
    decoded = eval_tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
    results = []
    for text in decoded:
        if "</think>" in text:
            text = text.partition("</think>")[2]
        text = postprocess_output(text.strip().replace("\n", " "))
        results.append(text)
    return results


print("Running inference on validation set …")
val_translits   = df_val["transliteration"].tolist()
val_references  = df_val["translation"].astype(str).str.strip().str.replace("\n", " ").tolist()

val_predictions = []
for i in tqdm(range(0, len(val_translits), EVAL_BATCH_SIZE)):
    batch = val_translits[i : i + EVAL_BATCH_SIZE]
    val_predictions.extend(_translate_batch(batch))

# ----- BLEU -----
bleu_metric = BLEU(effective_order=True)
bleu_score  = bleu_metric.corpus_score(val_predictions, [val_references]).score  # 0–100 scale

# ----- chrF++ -----
chrf_metric = CHRF(word_order=2)         # word_order=2 → chrF++
chrf_score  = chrf_metric.corpus_score(val_predictions, [val_references]).score  # 0–100 scale

# ----- Geometric Mean -----
geo_mean = math.sqrt(bleu_score * chrf_score)

print("\n========== Validation Scores ==========")
print(f"  BLEU   : {bleu_score:.4f}")
print(f"  chrF++ : {chrf_score:.4f}")
print(f"  GeoMean: {geo_mean:.4f}")
print("========================================")

# Log to wandb if session is active
try:
    wandb.log({
        "val/bleu":     bleu_score,
        "val/chrf++":   chrf_score,
        "val/geo_mean": geo_mean,
    })
except Exception:
    pass

Loading and preparing validation data …
--- Loading merged model for evaluation ---


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Running inference on validation set …


100%|██████████| 4/4 [01:40<00:00, 25.03s/it]


========== Validation Scores ==========
  BLEU   : 15.5418
  chrF++ : 42.5048
  GeoMean: 25.7021


# 2.推論

In [ ]:
import os
import pandas as pd
import torch
import re
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor

# =========================================================
# Configuration (推論ノートと同様)
# =========================================================
MODEL_DIR  = "/content/drive/MyDrive/kaggle/Akkadian/models/qwen3-4b-lora-deeppast-v1/merged"  # ← mergedモデルの保存先に合わせて変更（例: "/kaggle/working/qwen.../merged"）
TEST_CSV   = "/content/drive/MyDrive/kaggle/Akkadian/data/test.csv"
OUTPUT_CSV = "/content/drive/MyDrive/kaggle/Akkadian/results/submission.csv"
BATCH_SIZE = 8

In [ ]:
if __name__ == "__main__":
    test_cases = [
        "i-Si2-ba-at x x",
        "KÙ.B.-pi3-a",
        "test (break) more text",
        "test (large break) more text",
        "test (3 broken lines) more text",
        "a-na x [x] … test",
        "<gap> test <gap>",
    ]

    print("Transliteration Normalization Test Results:")
    print("=" * 50)

    for test in test_cases:
        normalized = normalize_transliteration(test)
        print(f"Input:      {test}")
        print(f"Normalized: {normalized}")
        print("-" * 30)

Transliteration Normalization Test Results:
Input:      i-Si2-ba-at x x
Normalized: i-Si2-ba-at <gap> <gap>
------------------------------
Input:      KÙ.B.-pi3-a
Normalized: KÙ.BABBAR-pi3-a
------------------------------
Input:      test (break) more text
Normalized: test <gap> more text
------------------------------
Input:      test (large break) more text
Normalized: test <gap> more text
------------------------------
Input:      test (3 broken lines) more text
Normalized: test <gap> more text
------------------------------
Input:      a-na x [x] … test
Normalized: a-na <gap> <gap> <gap> test
------------------------------
Input:      <gap> test <gap>
Normalized: <gap> test <gap>
------------------------------


In [ ]:
# =========================================================
# 1) Load Test Data
# =========================================================
print("--- Loading Test Data ---")
test_df = pd.read_csv(TEST_CSV)

# =========================================================
# 2) Initialize Processor/Tokenizer (推論ノートと同様)
# =========================================================
print("--- Initializing Transformers ---")
processor = AutoProcessor.from_pretrained(MODEL_DIR, trust_remote_code=True)
try:
    tokenizer = processor.tokenizer
except:
    tokenizer = processor

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =========================================================
# 3) Build prompts with chat template (推論ノートと同様)
# =========================================================
messages_list = [
    [{"role": "user", "content": f"/no_think Translate Akkadian to English: {preprocess_input(text)}"}]
    for text in test_df["transliteration"]
]

raw_prompts = [
    processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    for messages in messages_list
]

# =========================================================
# 4) Load merged model (推論ノートと同様)
#   NOTE: deep-past-qwen-lora-v2.ipynb は torch_dtype=torch.float32 を使用
# =========================================================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    device_map="auto",
    torch_dtype=torch.float32,  # ←ノート準拠（VRAM厳しければ float16/bfloat16 に変更）
    trust_remote_code=True,
)
model.eval()

# =========================================================
# 5) Stop token IDs (推論ノートと同様)
# =========================================================
eos_id = tokenizer.eos_token_id
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

stop_ids = [eos_id]
if im_end_id is not None:
    stop_ids.append(im_end_id)

print(f"Stopping on token IDs: {stop_ids}")

# =========================================================
# 6) Inference loop (推論ノートと同様)
# =========================================================
predictions = []

for i in tqdm(range(0, len(raw_prompts), BATCH_SIZE)):
    batch_prompts = raw_prompts[i : i + BATCH_SIZE]

    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            top_k=20,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=stop_ids,
        )

    input_len = inputs.input_ids.shape[1]
    generated_tokens = outputs[:, input_len:]
    decoded_batch = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    predictions.extend(decoded_batch)

# =========================================================
# 7) Postprocess (推論ノートと同様)
# =========================================================
final_predictions = []
for text in predictions:
    if "</think>" in text:
        text = text.partition("</think>")[2]
    text = postprocess_output(text.strip().replace(r"\n", " "))
    final_predictions.append(text)

# =========================================================
# 8) Save submission.csv (推論ノートと同様)
# =========================================================
submission_df = pd.DataFrame({"id": test_df["id"], "translation": final_predictions})
submission_df.to_csv(OUTPUT_CSV, index=False)
print(f"Submission saved to {OUTPUT_CSV}")
submission_df.head()

--- Loading Test Data ---
--- Initializing Transformers ---


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Stopping on token IDs: [151645, 151645]



100%|██████████| 1/1 [00:04<00:00,  4.93s/it]

Submission saved to /content/drive/MyDrive/kaggle/Akkadian/results/submission.csv


,id,translation
0,0,"From the city Kanesh, to Aaqil <gap> month We ..."
1,1,"When they arrived at the City from the south, ..."
2,2,"As for the man of the City, he has taken his a..."
3,3,The man's servants have arrived with the carav...
